In [672]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import emcee
from find_source import summary
import corner
import math
from astropy.coordinates import Angle
import astropy.units as units
import scipy.special as sp
import warnings

In [ ]:
def p_model(p_params, u, v):
    i0, l0, m0 = p_params
    return i0 * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def c_model(c_params, u, v):
    s0, l0, m0, vis_sigma = c_params
    return s0 * np.exp(-0.5*(u**2 + v**2)/vis_sigma**2) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def g_model(g_params, u, v):
    s0, l0, m0, vis_sigma, ratio, vis_theta = g_params
    return s0 * np.exp(-0.5*((u*np.cos(vis_theta)-v*np.sin(vis_theta))**2 + (u*np.sin(vis_theta)+v*np.cos(vis_theta))**2/ratio**2)/vis_sigma**2) \
            * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def d_model(d_params, u, v):
    s0, l0, m0, vis_r = d_params
    return s0 * 2*vis_r/np.sqrt(u**2+v**2) * sp.j1(np.sqrt(u**2+v**2)/vis_r) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

In [ ]:
def p_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 3))
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
    return p0

def c_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 4))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
    return p0

def g_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 6))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
        p0[i,4] = np.random.uniform(0, 1)
        p0[i,5] = np.random.uniform(-np.pi/2, np.pi/2)
    return p0

def d_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 4))
    source_area = rad_barea * total_flux / peak
    r = np.sqrt(source_area / (2*np.pi))
    vis_r = 1/(math.pi*r)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_r, 1.05*vis_r)
    return p0

In [ ]:
def p_prior(params):
    i0, l0, m0 = params
    if 0 < i0:
        return 0.0
    return -np.inf

def c_prior(params):
    s0, l0, m0, vis_sigma = params
    if 0 < s0 and 0 < vis_sigma:
        return 0.0
    return -np.inf

def g_prior(params):
    s0, l0, m0, vis_sigma, ratio, vis_theta = params
    if 0 < s0 and 0 < vis_sigma and 0 < ratio <= 1 and -np.pi/2 <= vis_theta <= np.pi/2:
        return 0.0
    return -np.inf

def d_prior(params):
    s0, l0, m0, vis_r = params
    if 0 < s0 and 0 < vis_r:
        return 0.0
    return -np.inf

In [ ]:
def log_likelihood(model, re, im, u, v, w):
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

In [ ]:
def log_probability(sources, params, re, im, u, v, w):

    # TODO: check this code
    log_prior = 0.0
    model = 0.0
    idx = 0
    for source in sources:
        n_params, _, prior_func, model_func = SOURCE_TYPES[source]
        source_params = params[idx:idx+n_params]
        lp = prior_func(source_params)
        if not np.isfinite(lp):
            return -np.inf
        log_prior += lp
        model += model_func(source_params, u, v)
        idx += n_params
    log_likelihood_value = log_likelihood(model, re, im, u, v, w)
    return log_prior + log_likelihood_value

In [ ]:
SOURCE_TYPES = {'p': [3, p_p0, p_prior, p_model], \
                'c': [4, c_p0, c_prior, c_model], \
                'g': [6, g_p0, g_prior, g_model], \
                'd': [4, d_p0, d_prior, d_model]} # TODO: finish filling in